# Problem 030:  Digit Fifth Powers

> Surprisingly there are only three numbers that can be written as the sum of fourth powers of their digits:
> $$\begin{align}
1634 &= 1^4 + 6^4 + 3^4 + 4^4\\
8208 &= 8^4 + 2^4 + 0^4 + 8^4\\
9474 &= 9^4 + 4^4 + 7^4 + 4^4
\end{align}$$
>
> As $1 = 1^4$ is not a sum it is not included.
>
> The sum of these numbers is $1634 + 8208 + 9474 = 19316$.
>
> Find the sum of all the numbers that can be written as the sum of fifth powers of their digits.


## Naive solution $\mathcal O\left(N^2\, 9^N\right)$

An easy way to code this is to just check for every number whether or not the sum of digit powers is equal to the value.  The two things that need to be developed is the code to do the check and finding the maximum number to do the search up to.

The `is_sum_of_digit_weights` is a one line checker for whether or not the sum of digit weights is equal to the input itself by parsing the `int` as a `str` and converting each `char` back to an `int`.  The use of the `wgt` array allows for pre-calculation of the powers of each digit so they are not calculated each time.

As for the largest possible value to check, consider that the largest sum of digit powers for a $n$-digit number will be a set of $n$ $9$'s.  At some length $n$, the largest sum of $n 9^N$ will be less than any $n$-digit number, at which point only values below $n9^N$ needs to be considered.  For small $N$, this occurs at $n = N + 1$.  This holds until about $N \simeq 30$, at which point the value of $n$ does decrease, but the calculation will be insanely long to worry about the few percent time difference of lowering the value.

For time scaling, there are $(N+1)9^N$ numbers being checked, and the check will require about $N$ calculations in the sum, hence a time complexity of $\mathcal O(N^2\,9^N)$.


In [1]:
%%time

def is_sum_of_digit_weights(x: int, wgt: list[int]) -> bool:
    return x == sum([wgt[int(a)] for a in str(x)])

def p030_naive(N: int) -> int:
    pown = [pow(i,N) for i in range(10)]
    imax = (N+1) * pown[9]
    return sum([i for i in range(10, imax) if is_sum_of_digit_weights(i, pown)])

N = 5
ans = p030_naive(N)

print(ans)

443839
CPU times: user 669 ms, sys: 809 μs, total: 670 ms
Wall time: 686 ms


## A faster approach $\mathcal O\left(N^{10}\ln N\right)$

The exponential time scaling above can be greatly improved to extend the problem to larger $N$.

The key idea used here is that any permutation of a set of digits will have the same sum of digit powers.  Thus, it is unnecessary to perform the check for all permutations of a set of digits.  Using `itertools.combination_with_replacement` can generate every set of digits to within a permutation.  Then, given a set of $N+1$ digits, the digit power sum is calculated.  If the digits in the sum are the same digits in the original number, not counting $0$'s, then the sum is a valid number.  To do this check, the sum is cast into a `str`, sorted, and then re-casted into an `int`.  The combination is also converted to a `str` and then an `int`, the digits already being in order.  This process conviently removes the $0$'s from the numbers to be checked.

The number of combinations with replacement for ten digits is equal to $9+N \choose 9$, which asymptotically goes as $N^9$.  Add the fact that each check requires a sort of $N+1$ values, that makes the overall time complexity $\mathcal O(N^{10}\ln N)$.  The first algorithm takes more than a minute to run $N=7$, whereas the code below can run $N=19$ in the same amount of time.

In [3]:
%%time

from itertools import combinations_with_replacement

def p030(N: int) -> int:
    val = 0
    pown = [pow(i,N) for i in range(10)]

    for x in combinations_with_replacement(range(10), N+1):
        y = sum(pown[a] for a in x)
        yp = int(''.join(sorted(str(y))))
        x = int(''.join(map(str,x)))
        if x == yp and y > 9:
            val += y
    
    return val

N = 5
ans = p030(N)

print(ans)

443839
CPU times: user 32.3 ms, sys: 984 μs, total: 33.3 ms
Wall time: 32.7 ms
